# Session 4 — Transient CFD and Spectral Analysis

**Post-CFD Analysis with Python | Dr. Nuha Aljuneidi**

Unsteady CFD produces a time history, not a single number. Before you can report a mean force or a shedding frequency, you must confirm the signal has reached statistical stationarity, then extract frequency content correctly. This session covers both.

## Learning outcomes
- Assess whether a transient force signal has reached statistical stationarity, using the trend across all analysis windows, not just the last two.
- Compute a frequency spectrum from a time signal using the FFT.
- Identify a dominant shedding frequency and quantify its uncertainty from the spectral peak width — not just the FFT bin resolution.
- Compute a Strouhal number and interpret it physically.

## Using your own Fluent or CSV data
This notebook uses synthetic data so you can run every cell immediately without a CFD license. When you are ready to use your own results, export a CSV from Fluent (or any solver) with coordinates, variable names, units, operating conditions, and a case identifier, then replace the synthetic-data cell below with:

```python
df = pd.read_csv("your_export.csv")
```

Map your solver's column names to the ones used in this notebook before continuing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.fft import rfft, rfftfreq

rng = np.random.default_rng(21)
print("Environment ready.")

## 1. A synthetic transient force history

Vortex shedding behind a bluff body produces an oscillating lift force. We synthesize a lift-coefficient time history with an initial transient (startup), a dominant shedding frequency, a weaker harmonic, and broadband turbulent noise — a realistic shape for a monitored Fluent force report.

In [ ]:
dt_s = 0.001
n_steps = 6000
t_s = np.arange(n_steps) * dt_s

f_shed_Hz = 45.0
startup_envelope = 1 - np.exp(-t_s / 0.4)
Cl_signal = (0.9 * np.sin(2 * np.pi * f_shed_Hz * t_s)
             + 0.15 * np.sin(2 * np.pi * 2 * f_shed_Hz * t_s + 0.3)
             + rng.normal(0, 0.05, n_steps)) * startup_envelope

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(t_s, Cl_signal, linewidth=0.8)
ax.set_xlabel("time (s)")
ax.set_ylabel("Cl (-)")
ax.set_title("Monitored lift-coefficient history")
plt.tight_layout()
plt.show()

## 2. Statistical stationarity

A signal is usable for spectral or mean-value analysis only after the initial transient has decayed. Check this by comparing windowed mean and variance over the back half of the record against the front half.

In [ ]:
def stationarity_check(signal, t, window_frac=0.2):
    """Fit a trend across ALL window means (not just the last two) and compare its total
    drift over the record to the noise level (mean window std) to flag remaining transients."""
    n = len(signal)
    w = int(n * window_frac)
    n_windows = n // w
    means, stds, window_t = [], [], []
    for i in range(n_windows):
        seg = signal[i*w:(i+1)*w]
        means.append(seg.mean())
        stds.append(seg.std())
        window_t.append(t[i*w])
    means, stds, window_t = np.array(means), np.array(stds), np.array(window_t)

    slope, _ = np.polyfit(window_t, means, 1)
    trend_drift = abs(slope) * (window_t[-1] - window_t[0])
    noise_level = stds.mean()
    stationarity_ratio = trend_drift / noise_level if noise_level > 0 else np.inf

    print(f"{'window':>8} {'t_start_s':>10} {'mean':>8} {'std':>8}")
    for i, (m, s) in enumerate(zip(means, stds)):
        print(f"{i:8d} {t[i*w]:10.3f} {m:8.4f} {s:8.4f}")
    print(f"\nTrend across all {n_windows} windows: slope = {slope:.5f} /s, "
          f"total drift over record = {trend_drift:.4f} (Cl units)")
    print(f"Noise level (mean window std) = {noise_level:.4f}")
    print(f"Stationarity ratio (trend drift / noise level) = {stationarity_ratio:.2f} — "
          f"{'stationary enough to analyze' if stationarity_ratio < 0.5 else 'still transient; discard more startup data'}")
    return means, stds

means, stds = stationarity_check(Cl_signal, t_s)

### Checkpoint 1 — Trim the transient
Based on the windowed means above, choose a start time `t_start_s` that excludes the startup transient. Slice `Cl_signal` and `t_s` to keep only data after that time, and re-plot. How much of the record did you discard, and is that a problem for spectral resolution (see Section 3)?

In [ ]:
# TODO: set t_start_s, build Cl_stat and t_stat as the trimmed, stationary signal
t_start_s = 0.0
Cl_stat = Cl_signal
t_stat = t_s


## 3. Frequency spectrum via FFT

With a stationary signal, compute the amplitude spectrum using `scipy.fft.rfft`. The frequency resolution is $\Delta f = 1/T_{record}$ and the maximum resolvable frequency (Nyquist) is $f_{max} = 1/(2\,\Delta t)$ — both set hard limits on what you can detect.

In [ ]:
signal_for_fft = Cl_stat - Cl_stat.mean()  # remove DC offset before transforming
n = len(signal_for_fft)

freq_Hz = rfftfreq(n, d=dt_s)
amplitude = np.abs(rfft(signal_for_fft)) * 2 / n

record_length_s = n * dt_s
freq_resolution_Hz = 1 / record_length_s
nyquist_Hz = 1 / (2 * dt_s)
print(f"Record length: {record_length_s:.3f} s  ->  frequency resolution: {freq_resolution_Hz:.3f} Hz")
print(f"Sample interval: {dt_s*1000:.2f} ms   ->  Nyquist frequency: {nyquist_Hz:.1f} Hz")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(freq_Hz, amplitude)
ax.set_xlim(0, 150)
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Cl amplitude (-)")
ax.set_title("Lift spectrum")
plt.tight_layout()
plt.show()

peak_idx = np.argmax(amplitude[1:]) + 1  # skip the zero-frequency bin
f_dominant_Hz = freq_Hz[peak_idx]

# Peak-width-at-half-maximum: a real uncertainty estimate, distinct from the FFT bin resolution.
# It reflects how sharply the peak is defined (e.g. wider if the shedding frequency drifts in time),
# not just the grid spacing of the frequency axis.
half_max = amplitude[peak_idx] / 2
i_left = peak_idx
while i_left > 0 and amplitude[i_left] > half_max:
    i_left -= 1
i_right = peak_idx
while i_right < len(amplitude) - 1 and amplitude[i_right] > half_max:
    i_right += 1
f_uncertainty_Hz = 0.5 * (freq_Hz[i_right] - freq_Hz[i_left])

print(f"Dominant frequency: {f_dominant_Hz:.2f} Hz, peak half-width uncertainty: "
      f"±{f_uncertainty_Hz:.2f} Hz")
print(f"(bin resolution is ±{freq_resolution_Hz:.2f} Hz — a floor set by record length, not the same "
      f"thing as the peak-width uncertainty above; both are well inside the Nyquist limit of {nyquist_Hz:.0f} Hz)")

### Checkpoint 2 — Resolution matters
Recompute the spectrum using only the *first* 500 samples of `signal_for_fft` instead of the full record. How much does the frequency resolution degrade, and does the dominant-frequency estimate still land close to the full-record value? State the general rule this demonstrates about record length and spectral resolution.

In [ ]:
# TODO: repeat the FFT on signal_for_fft[:500] and compare the resulting frequency resolution
# and dominant frequency to the full-record result


## Windowing and a more robust spectral estimate

A raw (unwindowed) FFT of a finite-length signal implicitly assumes the signal repeats periodically outside the recorded window. If it does not, the discontinuity at the edges leaks energy into neighboring frequency bins (spectral leakage), which can widen or bias a peak estimate. Two standard fixes: taper the signal with a window function (e.g., a Hann window) before the FFT, or use Welch's method (`scipy.signal.welch`), which averages the periodogram over multiple overlapping, windowed segments for a smoother, lower-variance estimate — at the cost of frequency resolution, since each segment is shorter than the full record.

In [ ]:
from scipy.signal import welch

# Windowed FFT: taper with a Hann window before transforming, to reduce spectral leakage.
hann_window = np.hanning(n)
amplitude_windowed = np.abs(rfft(signal_for_fft * hann_window)) * 2 / np.sum(hann_window)

# Welch's method: average the periodogram over overlapping windowed segments for a smoother,
# lower-variance spectral estimate, trading away some frequency resolution.
f_welch_Hz, psd_welch = welch(signal_for_fft, fs=1/dt_s, window="hann", nperseg=1024, noverlap=512)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(freq_Hz, amplitude, label="raw FFT", alpha=0.6)
axes[0].plot(freq_Hz, amplitude_windowed, label="Hann-windowed FFT", alpha=0.8)
axes[0].set_xlim(0, 150)
axes[0].set_xlabel("Frequency (Hz)")
axes[0].set_ylabel("Cl amplitude (-)")
axes[0].set_title("Raw vs. windowed FFT")
axes[0].legend()

axes[1].semilogy(f_welch_Hz, psd_welch)
axes[1].set_xlim(0, 150)
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_ylabel("PSD (Cl^2 / Hz)")
axes[1].set_title("Welch PSD estimate")

plt.tight_layout()
plt.show()

f_dominant_welch_Hz = f_welch_Hz[np.argmax(psd_welch)]
welch_resolution_Hz = f_welch_Hz[1] - f_welch_Hz[0]
print(f"Dominant frequency — raw FFT: {f_dominant_Hz:.2f} Hz, Welch PSD: {f_dominant_welch_Hz:.2f} Hz")
print(f"Welch frequency resolution: {welch_resolution_Hz:.2f} Hz (coarser than the full-record FFT's "
      f"{freq_resolution_Hz:.2f} Hz — the trade is lower variance for worse resolution)")

## 4. Strouhal number

The Strouhal number nondimensionalizes shedding frequency:

$$St = \frac{f\,L}{U_\infty}$$

where $L$ is a characteristic length (e.g., cylinder diameter) and $U_\infty$ is the freestream velocity. For a circular cylinder in the sub-critical Reynolds-number range, $St \approx 0.2$ is the expected physical benchmark.

In [ ]:
L_m = 0.05
U_inf_mps = 15.0

St = f_dominant_Hz * L_m / U_inf_mps
print(f"St = {St:.3f}  (value, reference: f={f_dominant_Hz:.2f} Hz, L={L_m} m, U_inf={U_inf_mps} m/s)")
print(f"Benchmark for a circular cylinder: St ~ 0.2 -> "
      f"{'consistent with expected bluff-body shedding' if 0.15 < St < 0.25 else 'outside typical range — check L, U_inf, or f_dominant'}")

### Checkpoint 3 — Report the frequency result
Write the full engineering-standard result statement for the shedding frequency and Strouhal number: value, unit, reference condition ($L$, $U_\infty$), numerical-quality check (spectral resolution relative to the peak), physical interpretation, and one limitation (e.g., synthetic single-probe signal, not a full 3D unsteady solution).

## Graduate/Advanced Extension
Add a slow amplitude modulation to `Cl_signal` (e.g., multiply by `1 + 0.3*np.sin(2*np.pi*2*t_s)`) before computing the spectrum. Describe what new feature appears around the dominant peak (sidebands), and explain physically what kind of unsteady flow behavior this could represent (e.g., beating between two shedding modes).

## Exit ticket
In three sentences: explain why you must check stationarity before running an FFT, state the frequency resolution of the spectrum you computed today and why it matters, and describe one transient signal from your own work you would now analyze differently.

**Next:** Session 5 applies these signal-processing tools to turbulence intensity and wake diagnostics.